# Lightweight Mathematical Fingerprinting for Foundation Model Backdoor Detection and Supply-Chain Integrity Verification

**End-to-end implementation of the six-stage workflow and experimental validation.**

This notebook implements:
1. Dataset and artifact acquisition (CIFAR-10, CIFAR-100, optional GTSRB), paired clean/backdoor model training, and legitimate derivatives.
2. Layer-wise parameter summaries and fixed-probe activations.
3. **RIM-FP**: transformation-aware descriptors, weighted fusion, secret-seeded projection, and derivative-calibrated conformal acceptance.
4. **SABER**: activation profiling, robust/OAS covariance, unclipped Mahalanobis scoring, max-T conformal layer localization, and three-way suspicion.
5. **FALV / MALP**: provenance graph, transformation-specific calibration, parent identification, hash-chain commitments, and authorization-aware replica classification.
6. Evidence fusion and the Stage 7 metrics.

**Data integrity note:** datasets are downloaded from their public torchvision sources (not fabricated). Backdoor examples are created from real training images using an explicitly documented patch-trigger protocol so the experiment has known ground truth. The workflow does not claim these generated pairs are the official BackdoorBench release. It records provenance and attack settings in the output directory. For a study requiring the exact BackdoorBench attack splits/configurations, replace the acquisition adapter with the official repository files.

The default run is CPU-compatible and deliberately uses a compact CNN and configurable subset/epoch settings. Increase the settings for a full-scale experiment. The first run requires internet access to download datasets.

## 0. Environment and configuration

In [ ]:
import os, sys, json, time, copy, math, random, hashlib, hmac, platform, warnings
from pathlib import Path
from dataclasses import dataclass, asdict
from collections import defaultdict, deque
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
import torchvision
from torchvision import datasets, transforms
from sklearn.covariance import OAS, MinCovDet
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score, confusion_matrix,
                             roc_curve)
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ROOT = Path("./lightweight_fingerprint_outputs")
DATA_DIR = ROOT / "datasets"
ARTIFACT_DIR = ROOT / "artifacts"
REPORT_DIR = ROOT / "reports"
for p in [ROOT, DATA_DIR, ARTIFACT_DIR, REPORT_DIR]: p.mkdir(parents=True, exist_ok=True)

# CPU-friendly defaults; raise these for more statistically stable experiments.
TRAIN_LIMIT = 12000       # number of real CIFAR-10 training images
TEST_LIMIT = 2000
EPOCHS_CLEAN = 4
EPOCHS_BACKDOOR = 4
BATCH_SIZE = 128
LEARNING_RATE = 1e-3
NUM_WORKERS = 2 if os.name != "nt" else 0
PROBE_COUNT = 256
PROJECTION_DIM = 128
SECRET_SEED = 20261116    # keep private in deployment; use a protected secret in production
BACKDOOR_TARGET = 0
TRIGGER_SIZE = 4
TRIGGER_VALUE = 1.0
ALPHA = 0.05
DOWNLOAD_CIFAR100 = True
DOWNLOAD_GTSRB = False   # set True to acquire GTSRB as an additional source dataset

print("Python:", platform.python_version(), "| PyTorch:", torch.__version__,
      "| Torchvision:", torchvision.__version__, "| Device:", DEVICE)
print("Output:", ROOT.resolve())

## 1. Open-source dataset acquisition and model artifact collection

In [ ]:
# Real public datasets. CIFAR-10 is the executable benchmark used below.
# CIFAR-100 and GTSRB are acquired as additional dataset assets when enabled.
train_tf = transforms.Compose([transforms.ToTensor()])
test_tf = transforms.Compose([transforms.ToTensor()])

cifar_train_full = datasets.CIFAR10(root=str(DATA_DIR), train=True, download=True, transform=train_tf)
cifar_test_full  = datasets.CIFAR10(root=str(DATA_DIR), train=False, download=True, transform=test_tf)

if DOWNLOAD_CIFAR100:
    cifar100_train = datasets.CIFAR100(root=str(DATA_DIR), train=True, download=True, transform=train_tf)
    cifar100_test  = datasets.CIFAR100(root=str(DATA_DIR), train=False, download=True, transform=test_tf)
if DOWNLOAD_GTSRB:
    gtsrb_train = datasets.GTSRB(root=str(DATA_DIR), split="train", download=True, transform=train_tf)
    gtsrb_test  = datasets.GTSRB(root=str(DATA_DIR), split="test", download=True, transform=test_tf)

def deterministic_subset(dataset, limit, seed=SEED):
    n = min(limit, len(dataset))
    rng = np.random.default_rng(seed)
    idx = rng.permutation(len(dataset))[:n].tolist()
    return Subset(dataset, idx)

clean_train = deterministic_subset(cifar_train_full, TRAIN_LIMIT)
clean_test = deterministic_subset(cifar_test_full, TEST_LIMIT)
print("CIFAR-10 selected:", len(clean_train), "train /", len(clean_test), "test")
print("CIFAR-10 classes:", cifar_train_full.classes)

### 1.1 Create reproducible clean/backdoor pairs from real images

In [ ]:
class TriggeredDataset(Dataset):
    """Patch-trigger benchmark built from genuine CIFAR-10 images; trigger metadata is explicit."""
    def __init__(self, base, target=BACKDOOR_TARGET, trigger_size=TRIGGER_SIZE,
                 poison_fraction=1.0, seed=SEED, train_mode=False):
        self.base, self.target, self.trigger_size = base, target, trigger_size
        self.poison_fraction, self.train_mode = poison_fraction, train_mode
        rng = np.random.default_rng(seed)
        self.poison = rng.random(len(base)) < poison_fraction
    def __len__(self): return len(self.base)
    def __getitem__(self, i):
        x, y = self.base[i]
        x = x.clone()
        if self.poison[i]:
            x[:, -self.trigger_size:, -self.trigger_size:] = TRIGGER_VALUE
            y = self.target
        return x, int(y)

backdoor_train = TriggeredDataset(clean_train, poison_fraction=0.20, train_mode=True)
backdoor_test = TriggeredDataset(clean_test, poison_fraction=1.0, train_mode=False)

# Fixed clean probes and a separate trigger-agnostic stress collection.
probe_subset = deterministic_subset(cifar_test_full, min(PROBE_COUNT, len(cifar_test_full)))
probe_loader = DataLoader(probe_subset, batch_size=64, shuffle=False, num_workers=NUM_WORKERS)

### 1.2 Data provenance and attack configuration

In [ ]:
provenance = {
    "dataset": "CIFAR-10",
    "source": "torchvision.datasets.CIFAR10 (Toronto dataset mirror)",
    "downloaded_at_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "train_selected": len(clean_train), "test_selected": len(clean_test),
    "selection_seed": SEED,
    "attack": {"type": "bottom-right square patch", "size": TRIGGER_SIZE,
               "pixel_value": TRIGGER_VALUE, "target_class": BACKDOOR_TARGET,
               "training_poison_fraction": 0.20, "test_trigger_fraction": 1.0},
    "note": "Controlled reproducible trigger benchmark; not represented as official BackdoorBench splits."
}
with open(REPORT_DIR / "dataset_provenance.json", "w") as f: json.dump(provenance, f, indent=2)
display(pd.json_normalize(provenance))

## 2. Model preparation and reference feature extraction

In [ ]:
class CompactCNN(nn.Module):
    """Same-architecture model family for controlled layer correspondence."""
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((1,1))
        )
        self.classifier = nn.Linear(128, num_classes)
    def forward(self, x):
        z = self.features(x).flatten(1)
        return self.classifier(z)

def make_loader(ds, shuffle):
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle,
                      num_workers=NUM_WORKERS, pin_memory=(DEVICE.type=="cuda"))

def train_model(model, loader, epochs, label):
    model.to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss()
    history = []
    for ep in range(epochs):
        model.train(); total_loss=0; correct=0; count=0
        for xb,yb in loader:
            xb,yb=xb.to(DEVICE),yb.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            logits=model(xb); loss=criterion(logits,yb)
            loss.backward(); opt.step()
            total_loss += loss.item()*len(yb)
            correct += (logits.argmax(1)==yb).sum().item(); count += len(yb)
        row={"epoch":ep+1,"loss":total_loss/count,"accuracy":correct/count}
        history.append(row)
        print(f"{label} | epoch {ep+1}/{epochs} | loss={row['loss']:.4f} | acc={row['accuracy']:.4f}")
    return pd.DataFrame(history)

def evaluate_classifier(model, loader):
    model.eval(); ys=[]; ps=[]
    with torch.no_grad():
        for xb,yb in loader:
            logits=model(xb.to(DEVICE))
            ys.extend(yb.numpy().tolist()); ps.extend(logits.argmax(1).cpu().numpy().tolist())
    return {"accuracy":accuracy_score(ys,ps),
            "macro_f1":f1_score(ys,ps,average="macro",zero_division=0)}

clean_model = CompactCNN()
clean_history = train_model(clean_model, make_loader(clean_train, True), EPOCHS_CLEAN, "Trusted clean model")

In [ ]:
backdoor_model = CompactCNN()
backdoor_history = train_model(backdoor_model, make_loader(backdoor_train, True), EPOCHS_BACKDOOR,
                               "Backdoored candidate")
clean_test_loader = make_loader(clean_test, False)
trigger_test_loader = make_loader(backdoor_test, False)
print("Trusted clean test:", evaluate_classifier(clean_model, clean_test_loader))
print("Backdoor candidate clean test:", evaluate_classifier(backdoor_model, clean_test_loader))
print("Backdoor candidate triggered test:", evaluate_classifier(backdoor_model, trigger_test_loader))

### 2.1 Legitimate derivatives: fine-tuned, pruned and quantized checkpoints

In [ ]:
def clone_model(model):
    m=CompactCNN()
    m.load_state_dict(copy.deepcopy(model.state_dict()))
    return m

# Legitimate fine-tuning derivative: one low-learning-rate epoch on clean data.
legit_finetuned = clone_model(clean_model).to(DEVICE)
opt=torch.optim.AdamW(legit_finetuned.parameters(), lr=LEARNING_RATE*0.1)
legit_finetuned.train()
for xb,yb in make_loader(clean_train, True):
    xb,yb=xb.to(DEVICE),yb.to(DEVICE)
    opt.zero_grad(set_to_none=True); loss=F.cross_entropy(legit_finetuned(xb),yb)
    loss.backward(); opt.step()
    # bounded adaptation for a compact demonstration
    if getattr(legit_finetuned, "_steps", 0) >= 99: break
    legit_finetuned._steps = getattr(legit_finetuned, "_steps", 0)+1

legit_pruned = clone_model(clean_model)
with torch.no_grad():
    for name,p in legit_pruned.named_parameters():
        if p.ndim >= 2:
            threshold=torch.quantile(p.abs(),0.05)
            p[p.abs()<threshold]=0

legit_quantized = clone_model(clean_model)
with torch.no_grad():
    for p in legit_quantized.parameters():
        scale=p.abs().max().clamp_min(1e-8)/127
        p.copy_(torch.round(p/scale).clamp(-127,127)*scale)

models = {
    "trusted_reference": clean_model.to(DEVICE),
    "legitimate_finetuned": legit_finetuned.to(DEVICE),
    "legitimate_pruned": legit_pruned.to(DEVICE),
    "legitimate_quantized": legit_quantized.to(DEVICE),
    "backdoored_candidate": backdoor_model.to(DEVICE)
}
for name,model in models.items():
    torch.save(model.state_dict(), ARTIFACT_DIR/f"{name}.pt")
print("Saved model artifacts:", sorted(p.name for p in ARTIFACT_DIR.glob("*.pt")))

### 2.2 Layer correspondence, parameter summaries and fixed-probe activations

In [ ]:
def compatible_layers(model):
    # Stable named parameter groups; compatible because all enrolled models share this architecture.
    return [(n,p) for n,p in model.named_parameters() if p.ndim >= 2]

def layer_parameter_summary(model):
    out=[]
    for name,p in compatible_layers(model):
        a=p.detach().float().cpu().numpy().ravel()
        out.extend([np.mean(a),np.std(a),np.mean(np.abs(a)),np.sqrt(np.mean(a*a)),
                    np.quantile(a,0.1),np.quantile(a,0.5),np.quantile(a,0.9)])
    return np.asarray(out,dtype=np.float64)

def collect_activations(model, loader, max_batches=None):
    model.eval(); feats=[]
    hook_handles=[]
    def hook_fn(module, inp, out):
        if isinstance(out, torch.Tensor):
            feats.append(out.detach().flatten(1).mean(1).cpu().numpy())
    # Capture output of each convolutional layer.
    for module in model.modules():
        if isinstance(module, nn.Conv2d):
            hook_handles.append(module.register_forward_hook(hook_fn))
    rows=[]
    with torch.no_grad():
        for bi,(xb,_) in enumerate(loader):
            feats.clear()
            model(xb.to(DEVICE))
            rows.append(np.stack([np.asarray(v) for v in feats],axis=1))
            if max_batches and bi+1>=max_batches: break
    for h in hook_handles: h.remove()
    return np.concatenate(rows,axis=0) if rows else np.empty((0,3))

def get_feature_bank(model_dict, loader):
    bank={}
    for name,model in model_dict.items():
        bank[name]={"params":layer_parameter_summary(model),
                    "acts":collect_activations(model,loader)}
    return bank

feature_bank=get_feature_bank(models,probe_loader)
print({k:(v["params"].shape,v["acts"].shape) for k,v in feature_bank.items()})

## 3. RIM-FP — Randomized Invariant Model Fingerprinting

The implementation uses per-layer scale normalization and sorted channel/filter statistics to reduce sensitivity to supported rescaling and channel-order changes. It then fuses standardized parameter and activation descriptors and applies a secret-seeded Gaussian projection. This is a practical approximate invariant descriptor, not a proof of invariance for arbitrary neural-network reparameterizations.

In [ ]:
def canonical_parameter_descriptor(model):
    desc=[]
    for name,p in compatible_layers(model):
        w=p.detach().float().cpu()
        # Normalize each output filter; sorted norms remove output-channel ordering effects.
        flat=w.reshape(w.shape[0],-1)
        norms=torch.linalg.vector_norm(flat,dim=1).clamp_min(1e-8)
        unit=flat/norms[:,None]
        filter_stats=torch.stack([
            unit.mean(1), unit.std(1), unit.abs().mean(1),
            torch.quantile(unit,0.25,dim=1), torch.quantile(unit,0.75,dim=1)
        ],dim=1).numpy()
        filter_stats=filter_stats[np.lexsort(filter_stats.T[::-1])]
        desc.extend(filter_stats.ravel().tolist())
        desc.extend([float(norms.mean()),float(norms.std()),float(torch.log(norms).mean())])
    return np.asarray(desc,dtype=np.float64)

def activation_descriptor(acts):
    a=np.asarray(acts,dtype=np.float64)
    return np.concatenate([a.mean(0),a.std(0),np.quantile(a,0.1,axis=0),
                           np.quantile(a,0.5,axis=0),np.quantile(a,0.9,axis=0)])

# Equal-length descriptors are guaranteed for the enrolled same-architecture family.
def fused_descriptor(model, probe_loader, w_param=0.6, w_act=0.4):
    p=canonical_parameter_descriptor(model)
    a=activation_descriptor(collect_activations(model,probe_loader))
    # Descriptor-level robust standardization avoids scale dominance.
    p=(p-np.median(p))/(np.median(np.abs(p-np.median(p)))+1e-8)
    a=(a-np.median(a))/(np.median(np.abs(a-np.median(a)))+1e-8)
    return np.concatenate([w_param*p,w_act*a])

raw_fused={name:fused_descriptor(m,probe_loader) for name,m in models.items()}
lengths={len(v) for v in raw_fused.values()}
assert len(lengths)==1, f"Architecture mismatch: descriptor lengths {lengths}"
rng=np.random.default_rng(SECRET_SEED)
input_dim=len(next(iter(raw_fused.values())))
R=rng.normal(0,1/np.sqrt(PROJECTION_DIM),(input_dim,PROJECTION_DIM))
def make_fingerprint(vec):
    z=np.asarray(vec)@R
    return z/(np.linalg.norm(z)+1e-12)

fingerprints={k:make_fingerprint(v) for k,v in raw_fused.items()}
reference_fp=fingerprints["trusted_reference"]
rim_distances={k:float(np.linalg.norm(v-reference_fp)) for k,v in fingerprints.items()}
display(pd.DataFrame({"model":list(rim_distances),"fingerprint_distance":list(rim_distances.values())}))

In [ ]:
# Legitimate-transformation conformal calibration: leave-one-derivative-out distances.
legit_names=["legitimate_finetuned","legitimate_pruned","legitimate_quantized"]
calibration_scores=np.array([rim_distances[n] for n in legit_names],dtype=float)
def conformal_upper_threshold(scores, alpha=ALPHA):
    s=np.sort(np.asarray(scores))
    rank=min(len(s),int(np.ceil((len(s)+1)*(1-alpha))))
    return float(s[max(0,rank-1)])
rim_threshold=conformal_upper_threshold(calibration_scores)
rim_results={}
for name,dist in rim_distances.items():
    # Empirical upper-tail conformal p-value with finite-sample correction.
    pval=(1+np.sum(calibration_scores>=dist))/(len(calibration_scores)+1)
    rim_results[name]={"distance":dist,"conformal_p":float(pval),
                       "consistent":bool(dist<=rim_threshold or name=="trusted_reference")}
print("RIM-FP threshold:",rim_threshold)
display(pd.DataFrame(rim_results).T)

## 4. SABER — Statistical Activation-Based Backdoor Examination and Recognition

Clean class-conditional probes and trigger-agnostic stress probes are profiled. Legitimate derivatives estimate expected activation variation. OAS covariance regularization supports small probe sets. The Mahalanobis distance is **not clipped**; max-T calibration controls family-wise layer localization. Three-way classification uses the calibrated global conformal p-value.

In [ ]:
def make_stress_loader(base_subset):
    # Trigger-agnostic stress: deterministic horizontal flip and mild brightness changes.
    class Stress(Dataset):
        def __init__(self,base): self.base=base
        def __len__(self): return len(self.base)
        def __getitem__(self,i):
            x,y=self.base[i]
            if i%2: x=torch.flip(x,dims=[2])
            x=(x* (0.9 if i%3==0 else 1.05)).clamp(0,1)
            return x,y
    return DataLoader(Stress(base_subset),batch_size=64,shuffle=False,num_workers=NUM_WORKERS)

stress_loader=make_stress_loader(probe_subset)
def activation_profile(model, loader):
    acts=collect_activations(model,loader)
    # Layer-wise moments: mean, std, 10th and 90th percentiles.
    return np.concatenate([acts.mean(0),acts.std(0),np.quantile(acts,.1,axis=0),
                           np.quantile(acts,.9,axis=0)])

profiles={}
for name,m in models.items():
    profiles[name]=np.concatenate([activation_profile(m,probe_loader),
                                    activation_profile(m,stress_loader)])

legit_profiles=np.stack([profiles[n] for n in legit_names])
# Robust center (coordinate-wise median); OAS covariance estimates legitimate variation.
profile_center=np.median(legit_profiles,axis=0)
profile_cov=OAS().fit(legit_profiles).covariance_
profile_inv=np.linalg.pinv(profile_cov + np.eye(profile_cov.shape[0])*1e-6)

def mahalanobis_unclipped(x, center=profile_center, inv_cov=profile_inv):
    d=np.asarray(x)-center
    return float(np.sqrt(max(0.0,d @ inv_cov @ d)))

saber_global={name:mahalanobis_unclipped(v) for name,v in profiles.items()}
print("SABER global activation deviation:")
display(pd.DataFrame({"model":list(saber_global),"global_score":list(saber_global.values())}))

In [ ]:
# Layer-level deviations. Each convolutional layer contributes four summary statistics
# for clean probes and four for stress probes.
n_layers=3
def layerwise_profile(model, loader):
    a=collect_activations(model,loader)
    return np.concatenate([a.mean(0),a.std(0),np.quantile(a,.1,axis=0),np.quantile(a,.9,axis=0)])
legit_layer_profiles=np.stack([
    np.concatenate([layerwise_profile(models[n],probe_loader),
                    layerwise_profile(models[n],stress_loader)]) for n in legit_names
])
layer_center=np.median(legit_layer_profiles,axis=0)
# Per-coordinate robust scale; aggregate into per-layer standardized absolute deviations.
layer_scale=1.4826*np.median(np.abs(legit_layer_profiles-layer_center),axis=0)+1e-5
def layer_statistics(model):
    v=np.concatenate([layerwise_profile(model,probe_loader),layerwise_profile(model,stress_loader)])
    z=np.abs((v-layer_center)/layer_scale)
    # 8 feature blocks (4 clean + 4 stress), each has n_layers entries.
    return np.array([np.max(z[j*n_layers:(j+1)*n_layers]) for j in range(8)])

legit_layer_stats=np.stack([layer_statistics(models[n]) for n in legit_names])
# Max-T calibration distribution from legitimate derivatives, with small-sample caveat.
cal_maxT=np.max(legit_layer_stats,axis=1)
maxT_threshold=conformal_upper_threshold(cal_maxT)
layer_scores={name:layer_statistics(m) for name,m in models.items()}
layer_localization={name:np.where(s>maxT_threshold)[0].tolist() for name,s in layer_scores.items()}
print("Max-T threshold:",maxT_threshold)
display(pd.DataFrame({"model":list(layer_localization),
                      "suspicious_profile_blocks":[str(x) for x in layer_localization.values()]}))

In [ ]:
# Global conformal p-value based on legitimate reference deviations.
legit_global=np.array([saber_global[n] for n in legit_names])
def upper_tail_pvalue(calibration, score):
    return float((1+np.sum(np.asarray(calibration)>=score))/(len(calibration)+1))
saber_results={}
for name,score in saber_global.items():
    p=upper_tail_pvalue(legit_global,score)
    if p <= ALPHA:
        status="suspicious"
    elif p >= 0.20:
        status="clean-consistent"
    else:
        status="inconclusive"
    saber_results[name]={"score":score,"conformal_p":p,"status":status,
                         "localized_blocks":layer_localization[name]}
display(pd.DataFrame(saber_results).T)

## 5. FALV / MALP — Fingerprint-Anchored Lineage Verification

In [ ]:
# Documented lineage graph: trusted -> legitimate derivatives.
# Backdoored candidate is enrolled as an unverified candidate with no authorized parent edge.
lineage = {
    "trusted_reference": {"parent":None,"transform":"root","authorized":True},
    "legitimate_finetuned":{"parent":"trusted_reference","transform":"fine_tune","authorized":True},
    "legitimate_pruned":{"parent":"trusted_reference","transform":"pruning","authorized":True},
    "legitimate_quantized":{"parent":"trusted_reference","transform":"quantization","authorized":True},
    "backdoored_candidate":{"parent":None,"transform":"unknown","authorized":False}
}
# Transformation-specific displacement calibration.
edge_calibration=defaultdict(list)
for name,meta in lineage.items():
    if meta["parent"] is not None:
        d=np.linalg.norm(fingerprints[name]-fingerprints[meta["parent"]])
        edge_calibration[meta["transform"]].append(float(d))
edge_thresholds={k:conformal_upper_threshold(v) for k,v in edge_calibration.items()}

In [ ]:
# Cryptographically anchored registry: content hashes and chained HMAC commitments.
REGISTRY_SECRET=os.environ.get("FALV_REGISTRY_SECRET","local-demo-secret-change-before-deployment").encode()
def state_dict_sha256(model):
    h=hashlib.sha256()
    for name,t in sorted(model.state_dict().items()):
        h.update(name.encode()); h.update(t.detach().cpu().contiguous().numpy().tobytes())
    return h.hexdigest()

registry={}
previous="GENESIS"
for name,model in models.items():
    digest=state_dict_sha256(model)
    record={"model":name,"sha256":digest,"parent":lineage[name]["parent"],
            "transform":lineage[name]["transform"],"authorized":lineage[name]["authorized"],
            "previous_commitment":previous}
    payload=json.dumps(record,sort_keys=True).encode()
    commitment=hmac.new(REGISTRY_SECRET,payload,hashlib.sha256).hexdigest()
    record["commitment"]=commitment
    registry[name]=record
    previous=commitment

def verify_registry_record(record):
    payload={k:v for k,v in record.items() if k!="commitment"}
    expected=hmac.new(REGISTRY_SECRET,json.dumps(payload,sort_keys=True).encode(),hashlib.sha256).hexdigest()
    return hmac.compare_digest(expected,record["commitment"])

integrity={n:verify_registry_record(r) and (state_dict_sha256(models[n])==r["sha256"])
           for n,r in registry.items()}
print("Registry integrity:",integrity)
with open(REPORT_DIR/"provenance_registry.json","w") as f: json.dump(registry,f,indent=2)

In [ ]:
# Open-set parent identification: compare candidate fingerprint to all enrolled models.
def identify_parent(candidate_name):
    dists={n:float(np.linalg.norm(fingerprints[candidate_name]-fingerprints[n]))
           for n in fingerprints if n!=candidate_name}
    ranked=sorted(dists.items(),key=lambda x:x[1])
    return ranked[0][0],ranked[0][1],ranked

falv_results={}
for name in models:
    parent,dist,ranking=identify_parent(name)
    meta=lineage[name]
    documented_ok=(meta["parent"] is None or
                    (parent==meta["parent"] and dist<=edge_thresholds.get(meta["transform"],np.inf)))
    authorized=bool(meta["authorized"])
    crypto_ok=bool(integrity[name])
    if name=="trusted_reference":
        status="trusted reference"
    elif authorized and documented_ok and crypto_ok:
        status="verified derivative"
    elif not authorized and saber_results[name]["status"]=="suspicious":
        status="suspicious derivative"
    else:
        status="unverified replica"
    falv_results[name]={"nearest_enrolled_parent":parent,"parent_distance":dist,
                        "documented_parent_compatible":documented_ok,
                        "authorized":authorized,"integrity_valid":crypto_ok,
                        "status":status,"ranking":ranking}
display(pd.DataFrame(falv_results).T)

## 6. Integrated model security decision

In [ ]:
def integrated_decision(name):
    if name=="trusted_reference":
        return "trusted reference"
    fp_ok=rim_results[name]["consistent"]
    saber=saber_results[name]["status"]
    falv=falv_results[name]["status"]
    auth=falv_results[name]["authorized"]
    integrity_ok=falv_results[name]["integrity_valid"]
    # Preserve evidence availability and avoid allowing a single weak signal to silently override others.
    if saber=="suspicious" and (not auth or not fp_ok):
        status="suspected backdoored model"
    elif falv=="verified derivative" and fp_ok and saber!="suspicious" and integrity_ok:
        status="verified legitimate derivative"
    elif not auth and falv=="unverified replica":
        status="unverified replica"
    else:
        status="inconclusive candidate"
    return status

integrated={n:integrated_decision(n) for n in models}
security_report=[]
for n in models:
    security_report.append({
        "model":n,"RIM_FP_distance":rim_results[n]["distance"],
        "RIM_FP_p":rim_results[n]["conformal_p"],
        "RIM_FP_consistent":rim_results[n]["consistent"],
        "SABER_score":saber_results[n]["score"],
        "SABER_p":saber_results[n]["conformal_p"],
        "SABER_status":saber_results[n]["status"],
        "FALV_status":falv_results[n]["status"],
        "authorization":falv_results[n]["authorized"],
        "registry_integrity":falv_results[n]["integrity_valid"],
        "integrated_security_status":integrated[n]
    })
security_df=pd.DataFrame(security_report)
display(security_df)
security_df.to_csv(REPORT_DIR/"integrated_security_report.csv",index=False)

## 7. Experimental analysis and mathematical validation

### 7.1 RIM-FP metrics

In [ ]:
# Pairwise identity scores: genuine pairs are reference-to-legitimate derivatives;
# impostor pairs are reference-to-backdoored candidate and cross-model negatives.
genuine_scores=np.array([rim_distances[n] for n in legit_names])
impostor_scores=np.array([rim_distances["backdoored_candidate"]])
all_scores=np.concatenate([genuine_scores,impostor_scores])
y_pair=np.concatenate([np.ones(len(genuine_scores)),np.zeros(len(impostor_scores))])
# Similarity is negative distance.
sim=-all_scores
fpr,tpr,thresholds=roc_curve(y_pair,sim)
fnr=1-tpr
eer_idx=np.nanargmin(np.abs(fpr-fnr))
eer=float((fpr[eer_idx]+fnr[eer_idx])/2)
rim_metrics={
    "EER":eer,"TPR_at_threshold":float(np.mean(genuine_scores<=rim_threshold)),
    "FPR_at_threshold":float(np.mean(impostor_scores<=rim_threshold)),
    "Fingerprint_Collision_Rate":float(np.mean(impostor_scores<=1e-8)),
    "Fingerprint_Stability_Score":float(np.mean(np.exp(-genuine_scores))),
    "Fingerprint_Size_bytes":int(next(iter(fingerprints.values())).nbytes)
}
display(pd.Series(rim_metrics,name="RIM-FP"))

### 7.2 SABER model-level detection metrics

In [ ]:
# Ground truth is derived from the controlled artifact construction, not inferred from model outputs.
y_true=np.array([0,0,0,0,1])  # last model is the deliberately backdoored artifact
# Convert calibrated suspicion status into binary predictions.
y_pred=np.array([int(saber_results[n]["status"]=="suspicious") for n in models])
# Continuous score supports AUROC/AUPRC.
y_score=np.array([saber_results[n]["score"] for n in models])
saber_metrics={
    "Accuracy":accuracy_score(y_true,y_pred),
    "Precision":precision_score(y_true,y_pred,zero_division=0),
    "Recall_Detection_Rate":recall_score(y_true,y_pred,zero_division=0),
    "F1_Score":f1_score(y_true,y_pred,zero_division=0),
    "AUROC":roc_auc_score(y_true,y_score) if len(np.unique(y_true))>1 else np.nan,
    "AUPRC":average_precision_score(y_true,y_score),
    "False_Alarm_Rate":float(np.mean(y_pred[y_true==0])) if np.any(y_true==0) else np.nan,
    "Suspicious_Layer_Localization_Accuracy":np.nan # ground-truth modified layer is not known for this training run
}
display(pd.Series(saber_metrics,name="SABER"))
print("Confusion matrix [clean, backdoored]:\n",confusion_matrix(y_true,y_pred))

### 7.3 MALP lineage metrics and integrated metrics

In [ ]:
# Ground-truth parent labels exist for documented derivatives in this controlled registry.
parent_pairs=[n for n in models if lineage[n]["parent"] is not None]
parent_correct=[falv_results[n]["nearest_enrolled_parent"]==lineage[n]["parent"] for n in parent_pairs]
valid_histories=[n for n in models if lineage[n]["authorized"]]
malp_metrics={
    "Parent_Identification_Accuracy":float(np.mean(parent_correct)) if parent_correct else np.nan,
    "Lineage_Reconstruction_Accuracy":float(np.mean(parent_correct)) if parent_correct else np.nan,
    "Provenance_Verification_Rate":float(np.mean([falv_results[n]["integrity_valid"] for n in valid_histories])) if valid_histories else np.nan,
    "Unauthorized_Replica_Detection_Rate":float(falv_results["backdoored_candidate"]["status"] in ["unverified replica","suspicious derivative"]),
    "False_Rejection_Rate":float(np.mean([falv_results[n]["status"]!="verified derivative" for n in parent_pairs])) if parent_pairs else np.nan,
    "Lineage_Consistency_Score":float(np.mean(parent_correct)) if parent_correct else np.nan
}
display(pd.Series(malp_metrics,name="MALP"))

# For the integrated classification benchmark, map artifacts to expected security categories.
expected={
 "trusted_reference":"trusted reference",
 "legitimate_finetuned":"verified legitimate derivative",
 "legitimate_pruned":"verified legitimate derivative",
 "legitimate_quantized":"verified legitimate derivative",
 "backdoored_candidate":"suspected backdoored model"
}
labels=list(expected)
yt=[expected[n] for n in labels]
yp=[integrated[n] for n in labels]
integrated_metrics={
    "Overall_Classification_Accuracy":accuracy_score(yt,yp),
    "Macro_F1":f1_score(yt,yp,average="macro",zero_division=0),
    "Security_Detection_Rate":float(np.mean([yp[i] in ["suspected backdoored model","unverified replica"] for i,n in enumerate(labels) if n=="backdoored_candidate"])),
    "False_Positive_Rate":float(np.mean([yp[i] in ["suspected backdoored model","unverified replica"] for i,n in enumerate(labels) if n!="backdoored_candidate"])),
    "Decision_Agreement":float(np.mean(np.array(yt)==np.array(yp)))
}
display(pd.Series(integrated_metrics,name="Integrated"))

### 7.4 Lightweight computational performance

In [ ]:
# Measure fingerprint and end-to-end candidate assessment latency on the current runtime.
candidate=models["backdoored_candidate"]
t0=time.perf_counter()
_ = fused_descriptor(candidate,probe_loader)
_ = make_fingerprint(fused_descriptor(candidate,probe_loader))
fingerprint_time=time.perf_counter()-t0

t0=time.perf_counter()
_ = activation_profile(candidate,probe_loader)
_ = mahalanobis_unclipped(profiles["backdoored_candidate"])
_ = identify_parent("backdoored_candidate")
detection_latency=(time.perf_counter()-t0)*1000

import tracemalloc
tracemalloc.start()
_ = canonical_parameter_descriptor(candidate)
current,peak=tracemalloc.get_traced_memory()
tracemalloc.stop()

fp_bytes=next(iter(fingerprints.values())).nbytes
computational_metrics={
    "Fingerprint_Generation_Time_seconds":fingerprint_time,
    "Detection_Latency_milliseconds":detection_latency,
    "Peak_Python_Allocation_MB":peak/(1024**2),
    "Fingerprint_Storage_KB":fp_bytes/1024,
    "Throughput_models_per_second":1000/max(detection_latency,1e-9),
    "Fingerprint_Dimension":len(reference_fp),
    "Parameter_Descriptor_Dimension":input_dim
}
display(pd.Series(computational_metrics,name="Efficiency"))

## 8. Results visualization and export

In [ ]:
metric_groups={
    "RIM-FP":rim_metrics,
    "SABER":saber_metrics,
    "MALP":malp_metrics,
    "Integrated":integrated_metrics,
    "Efficiency":computational_metrics
}
flat=[]
for group,metrics in metric_groups.items():
    for metric,value in metrics.items():
        if isinstance(value,(int,float,np.integer,np.floating)) and np.isfinite(value):
            flat.append({"Module":group,"Metric":metric,"Value":float(value)})
metrics_df=pd.DataFrame(flat)
metrics_df.to_csv(REPORT_DIR/"all_experimental_metrics.csv",index=False)
display(metrics_df)

plt.figure(figsize=(10,4))
plt.bar(list(rim_distances.keys()),list(rim_distances.values()))
plt.xticks(rotation=25,ha="right"); plt.ylabel("Euclidean fingerprint distance")
plt.title("RIM-FP fingerprint displacement"); plt.tight_layout()
plt.savefig(REPORT_DIR/"rim_fp_distances.png",dpi=180); plt.show()

plt.figure(figsize=(8,4))
plt.bar(list(saber_global.keys()),list(saber_global.values()))
plt.xticks(rotation=25,ha="right"); plt.ylabel("Unclipped Mahalanobis score")
plt.title("SABER activation deviation"); plt.tight_layout()
plt.savefig(REPORT_DIR/"saber_scores.png",dpi=180); plt.show()

print("Reports written to:", REPORT_DIR.resolve())

## 9. Reproducibility, limitations, and extension points

- **No fabricated input records:** the image samples come from downloaded CIFAR-10/CIFAR-100/GTSRB datasets. The backdoor trigger is a controlled transformation applied to genuine images, with its settings saved in `dataset_provenance.json`.
- **BackdoorBench compatibility:** this notebook follows the requested CIFAR-10/CIFAR-100/GTSRB acquisition direction, but the executable demonstration uses a reproducible patch protocol rather than importing every official BackdoorBench attack implementation. For benchmark-grade comparison, add the official attack-specific poison indices/configurations and preserve their labels.
- **Foundation-model scale:** the compact CNN is used to make the complete workflow runnable on CPU. For a production foundation-model audit, replace `CompactCNN` with the exact architecture/checkpoint loader and ensure the reference/candidate layers have validated correspondence.
- **Invariant fingerprinting:** filter normalization and sorting address selected rescaling/channel-order effects; arbitrary neuron permutations across coupled layers require graph-aware canonicalization and cannot be guaranteed by this lightweight approximation.
- **Conformal calibration:** only three legitimate derivatives are generated here, so p-values and thresholds are coarse. Use many independent authorized derivatives and a held-out calibration set for meaningful statistical guarantees.
- **SABER labels:** the notebook has known artifact-level clean/backdoor ground truth. The suspicious-layer localization accuracy is left `NaN` because the exact internal layer(s) causally responsible for the backdoor are not independently annotated.
- **Registry security:** the HMAC secret is a local demonstration secret unless `FALV_REGISTRY_SECRET` is set. Production deployments should use a protected signing key/KMS, immutable append-only storage, signed model manifests, and independently verified authorization records.
- **Do not interpret sample metrics as guaranteed targets:** the notebook computes observed results. It does not hard-code accuracy or claim a detection result in advance.

### Output files
`lightweight_fingerprint_outputs/` contains downloaded datasets, model checkpoints, `dataset_provenance.json`, `provenance_registry.json`, integrated security report, metric CSV, and plots.